In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.titleweight'] = 'bold'

DATA = 'data'

def to_dt(ms):
    return pd.to_datetime(pd.to_numeric(ms, errors='coerce'), unit='ms')

In [ ]:
labels = pd.read_csv(f'{DATA}/incident_label.csv')
labels['ts_start_dt'] = to_dt(labels['ts_start'])
labels['ts_end_dt'] = to_dt(labels['ts_end'])
labels['ts_end_f'] = labels['ts_end'].fillna(labels['ts_start'] + 3_600_000)
labels[['id', 'tier', 'kind', 'severity', 'provenance', 'ts_start_dt']].sort_values('ts_start_dt')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
colors = {1: '#d62728', 2: '#ff7f0e', 3: '#7f7f7f'}
for tier, grp in labels.groupby('tier'):
    is_lt = grp['kind'] == 'load_test'
    ax.scatter(grp.loc[~is_lt, 'ts_start_dt'], [tier]*(~is_lt).sum(),
               color=colors[tier], s=80, label=f'tier {tier}', zorder=3)
    ax.scatter(grp.loc[is_lt, 'ts_start_dt'], [tier]*is_lt.sum(),
               color=colors[tier], s=80, marker='x', zorder=3)
ax.scatter([], [], color='gray', marker='x', s=80, label='load_test (não-incidente)')
ax.set_yticks([1, 2, 3]); ax.set_ylim(0.5, 3.5)
ax.set_ylabel('tier'); ax.set_title(f'Os {len(labels)} rótulos do golden set, no tempo (X = teste de carga, não incidente)')
ax.legend(loc='upper left', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%b'))
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

print(f"Total: {len(labels)} linhas | {(labels['kind']!='load_test').sum()} incidentes reais | "
      f"{(labels['kind']=='load_test').sum()} load_test (excluir)")
print("\nDistribuição por provenance (confiabilidade do rótulo):")
print(labels['provenance'].value_counts())

In [ ]:
inc = pd.read_csv(f'{DATA}/incidents.csv')
inc['opened_dt'] = to_dt(inc['opened_at'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
inc['priority'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#4C72B0')
axes[0].set_title('Distribuição de prioridade (783 chamados brutos)')
axes[0].set_xlabel('priority'); axes[0].set_ylabel('qtde')

video_mask = inc['business_service'].fillna('').str.contains('Globoplay|[Vv]ídeo', regex=True)
relevant = inc[inc['priority'].isin(['P1', 'P2', 'P3']) & video_mask]
inc['relevante'] = 'ruído (P4/P5 ou não-vídeo)'
inc.loc[inc['priority'].isin(['P1','P2','P3']) & video_mask, 'relevante'] = 'P1-P3 + vídeo/Globoplay'
inc['relevante'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.0f%%', ylabel='')
axes[1].set_title('Quanto do chamado bruto é sinal de verdade')
plt.tight_layout(); plt.show()

print(f"P1-P3 + vídeo/Globoplay: {len(relevant)} chamados")
already = set(labels['id'].str.replace('itsm-', '', regex=False))
novos = sorted(set(relevant['sys_id']) - set())  # sys_id != number; comparar por número
novos_num = sorted(set(relevant['number']) - already)
print(f"Já estão no golden set: {len(set(relevant['number']) & already)}")
print(f"Ainda não rotulados (dá pra adicionar de graça, olhando o ticket): {len(novos_num)} -> {novos_num}")

In [ ]:
sb = pd.read_csv(f'{DATA}/signal_bucket.csv', dtype={'source': 'category', 'entity': 'category', 'metric': 'category'})
sb['dt'] = to_dt(sb['bts'])

fig, ax = plt.subplots(figsize=(11, 4))
sb['source'].value_counts().plot(kind='barh', ax=ax, color='#55A868')
ax.set_title(f'Volume de pontos por fonte ({len(sb):,} pontos, {sb["entity"].nunique()} entidades, {sb["metric"].nunique()} métricas)')
ax.set_xlabel('pontos')
plt.tight_layout(); plt.show()

In [ ]:
stats = sb.groupby(['source', 'entity', 'metric'], observed=True)['value'].agg(['mean', 'std', 'count'])
stats = stats[(stats['std'] > 1e-9) & (stats['count'] >= 30)]
sb2 = sb.merge(stats, on=['source', 'entity', 'metric'], how='inner')
sb2['z'] = (sb2['value'] - sb2['mean']) / sb2['std']

anomalias = sb2[sb2['z'].abs() > 4].copy().sort_values('z', ascending=False)
print(f"{len(stats)} séries analisadas | {len(anomalias)} pontos com |z|>4 ({len(anomalias)/len(sb2)*100:.3f}% do total)")
anomalias[['dt', 'source', 'entity', 'metric', 'value', 'z']].head(15)

In [ ]:
# Plot da série com o maior z-score isolado (mostra o método funcionando em dado real)
top = anomalias.iloc[0]
serie = sb2[(sb2['source']==top['source']) & (sb2['entity']==top['entity']) & (sb2['metric']==top['metric'])].sort_values('dt')

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(serie['dt'], serie['value'], color='#4C72B0', lw=1, label='valor observado')
m, s = serie['mean'].iloc[0], serie['std'].iloc[0]
ax.axhline(m, color='gray', ls='--', lw=1, label='baseline (média)')
ax.fill_between(serie['dt'], m-4*s, m+4*s, color='gray', alpha=0.15, label='faixa normal (±4σ)')
flag = serie[serie['z'].abs() > 4]
ax.scatter(flag['dt'], flag['value'], color='#d62728', s=40, zorder=5, label='anomalia (|z|>4)')

for _, row in labels.iterrows():
    if pd.isna(row['ts_start_dt']):
        continue
    end = row['ts_end_dt'] if pd.notna(row['ts_end_dt']) else row['ts_start_dt'] + pd.Timedelta(hours=1)
    ax.axvspan(row['ts_start_dt'], end, color='orange' if row['kind']=='incident' else 'green', alpha=0.15)

ax.set_title(f"{top['source']} | {top['entity']} | {top['metric']}  (laranja = janela de incidente rotulado, verde = load_test)")
ax.legend(loc='upper left', fontsize=8)
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
incidentes_reais = labels[labels['kind'] == 'incident'].copy()
incidentes_reais['ts_end_filled'] = incidentes_reais['ts_end'].fillna(incidentes_reais['ts_start'] + 3_600_000)
iidx = pd.IntervalIndex.from_arrays(incidentes_reais['ts_start'], incidentes_reais['ts_end_filled'], closed='both')

def bate_incidente(ts):
    try:
        return iidx.get_indexer([ts])[0] != -1
    except Exception:
        return False

amostra = anomalias.head(300).copy()  # amostra das maiores anomalias, checagem é O(log n) mas por segurança
amostra['dentro_de_incidente_conhecido'] = amostra['bts'].apply(bate_incidente)

resumo = amostra['dentro_de_incidente_conhecido'].value_counts()
print("Das 300 maiores anomalias:")
print(f"  dentro de janela já rotulada: {resumo.get(True, 0)}")
print(f"  FORA de qualquer janela rotulada (candidato novo): {resumo.get(False, 0)}")
print("\nExemplos de candidatos não rotulados (maior z primeiro):")
amostra[~amostra['dentro_de_incidente_conhecido']][['dt','source','entity','metric','value','z']].head(10)

In [ ]:
qoe = pd.read_csv(f'{DATA}/live_qoe_hourly.csv')
qoe['dt'] = to_dt(qoe['hour_ts'])
qoe_agg = qoe.groupby('dt', as_index=False)[['latency_ms_p95', 'bufferratio_avg', 'stalls_avg']].mean()

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(qoe_agg['dt'], qoe_agg['latency_ms_p95'], color='#4C72B0'); axes[0].set_title('Latência de início (p95, ms)')
axes[1].plot(qoe_agg['dt'], qoe_agg['bufferratio_avg'], color='#DD8452'); axes[1].set_title('Buffer ratio médio (fração do tempo travado)')
axes[2].plot(qoe_agg['dt'], qoe_agg['stalls_avg'], color='#C44E52'); axes[2].set_title('Travamentos médios por sessão')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
dev = pd.read_csv(f'{DATA}/device_hourly.csv')
dev_agg = dev.groupby('device', as_index=False)[['sessions', 'errors']].sum()
dev_agg = dev_agg[dev_agg['sessions'] > dev_agg['sessions'].quantile(0.5)]  # remove cauda de ruído/baixo volume
dev_agg['error_rate_pct'] = dev_agg['errors'] / dev_agg['sessions'] * 100
top = dev_agg.sort_values('error_rate_pct', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(top['device'], top['error_rate_pct'], color='#8172B2')
ax.set_xlabel('taxa de erro (%)'); ax.set_title('Top 15 dispositivos por taxa de erro (entre os de volume acima da mediana)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
BUSINESS_PATTERNS = ['rate', 'pct', 'latency', 'error', 'p50', 'p90', 'p95', 'p99',
                      'rebuffer', 'buffer', 'stall', 'rpm']
sb['metric_str'] = sb['metric'].astype(str)
sb['day'] = sb['dt'].dt.floor('D')
business_mask = sb['metric_str'].str.contains('|'.join(BUSINESS_PATTERNS), case=False, regex=True)
sbf = sb[business_mask].copy()
print(f"{len(sbf):,} de {len(sb):,} pontos após filtro de métricas de negócio "
      f"({sbf.groupby(['source','entity','metric'], observed=True).ngroups} séries)")

TRAILING_DAYS, WARMUP_DAYS, MIN_POINTS = 7, 3, 96

def baseline_walkforward(df):
    rows = []
    for (source, entity, metric), g in df.groupby(['source', 'entity', 'metric'], observed=True):
        days = sorted(g['day'].unique())
        if len(days) < WARMUP_DAYS + 1:
            continue
        by_day = {d: g.loc[g['day'] == d, 'value'].to_numpy() for d in days}
        for i, d in enumerate(days):
            if i < WARMUP_DAYS:
                continue
            window_days = [dd for dd in days if 1 <= (d - dd).days <= TRAILING_DAYS]
            if not window_days:
                continue
            vals = np.concatenate([by_day[dd] for dd in window_days])
            if len(vals) < MIN_POINTS:
                continue
            med = np.median(vals)
            mad = np.median(np.abs(vals - med)) * 1.4826
            if mad < 1e-9:
                continue
            rows.append((source, entity, metric, d, med, mad, max(window_days) < d))
    return pd.DataFrame(rows, columns=['source', 'entity', 'metric', 'day',
                                        'baseline_median', 'baseline_mad', 'sem_vazamento'])

baseline = baseline_walkforward(sbf)
assert baseline['sem_vazamento'].all(), "VAZAMENTO: alguma baseline usou dia >= dia pontuado"
print(f"{len(baseline):,} pares (série,dia) com baseline válida | teste de não-vazamento: OK "
      f"(100% das janelas usam só dias estritamente anteriores)")

sbf = sbf.merge(baseline.drop(columns='sem_vazamento'), on=['source', 'entity', 'metric', 'day'], how='inner')
sbf['z'] = (sbf['value'] - sbf['baseline_median']) / sbf['baseline_mad']

tier = pd.read_csv(f'{DATA}/cdn_entity_tier.csv')[['source', 'entity', 'tier']].drop_duplicates(['source', 'entity'])
sbf = sbf.merge(tier, on=['source', 'entity'], how='left')
sbf['tier'] = sbf['tier'].fillna(0).astype(int)
print(f"{len(sbf):,} pontos pontuáveis (walk-forward + robusto) após junção com tier de tráfego")


In [ ]:
import json as _json
from scipy import stats as spstats

def clopper_pearson(k, n, alpha=0.05):
    if n == 0: return (0.0, 1.0)
    lo = 0.0 if k == 0 else spstats.beta.ppf(alpha/2, k, n-k+1)
    hi = 1.0 if k == n else spstats.beta.ppf(1-alpha/2, k+1, n-k)
    return (lo, hi)

def wilson_ci(k, n, z=1.96):
    if n == 0: return (0.0, 1.0)
    phat = k/n
    denom = 1 + z**2/n
    center = (phat + z**2/(2*n)) / denom
    margin = z * ((phat*(1-phat)/n + z**2/(4*n**2))**0.5) / denom
    return (max(0.0, center-margin), min(1.0, center+margin))

sb_min, sb_max = sb['bts'].min(), sb['bts'].max()
warmup_cutoff = sb_min + WARMUP_DAYS * 86_400_000
print(f"Cobertura real do signal_bucket: {to_dt(pd.Series([sb_min])).iloc[0]} -> {to_dt(pd.Series([sb_max])).iloc[0]}")

overlap_mask = (labels['kind']=='incident') & (labels['ts_start'] <= sb_max) & (labels['ts_end_f'] >= warmup_cutoff)
fora_da_janela = labels[(labels['kind']=='incident') & ~overlap_mask]
print(f"\n{overlap_mask.sum()} de {(labels['kind']=='incident').sum()} incidentes reais caem dentro da "
      f"janela com telemetria (após os {WARMUP_DAYS}d de warm-up). Os outros "
      f"{len(fora_da_janela)} são de meses ANTES do período exportado em `signal_bucket`, e não têm "
      f"como ser detectados por nenhum método rodando só sobre esse dado, calibração nenhuma resolve isso.")
print("Provenance dos que caem fora da janela (pra contexto):",
      dict(fora_da_janela['provenance'].value_counts()))

confiaveis = labels[overlap_mask].reset_index(drop=True)
print(f"\n{len(confiaveis)} incidentes avaliáveis nesta rodada (dentro da janela de telemetria): "
      f"{list(confiaveis['id'])} | provenance: {dict(confiaveis['provenance'].value_counts())}")

# janelas "quietas": longe de qualquer rótulo (±2h) e de qualquer live_event
live_ev = pd.read_csv(f'{DATA}/live_event.csv')
buf = 7_200_000
excl_lo = (labels['ts_start']-buf).to_numpy()
excl_hi = (labels['ts_end'].fillna(labels['ts_start']+3_600_000)+buf).to_numpy()
live_lo = live_ev['start_ts'].to_numpy()
live_hi = live_ev['end_ts'].to_numpy()

def dentro_de_alguma(ts, los, his):
    return bool(((los <= ts) & (ts <= his)).any())

rng = np.random.default_rng(42)
t_min, t_max = int(sb['bts'].min()), int(sb['bts'].max())
quiet = []
tries = 0
while len(quiet) < 170 and tries < 20000:
    tries += 1
    s = int(rng.integers(t_min, t_max - 3_600_000))
    if dentro_de_alguma(s, excl_lo, excl_hi) or dentro_de_alguma(s, live_lo, live_hi):
        continue
    quiet.append((s, s + 3_600_000))
load_test = [(int(r['ts_start']), int(r['ts_end_f'])) for _, r in labels[labels['kind']=='load_test'].iterrows()]
negativos_calib, negativos_report = quiet[:50], quiet[50:]
print(f"{len(quiet)} janelas quietas geradas | {len(negativos_calib)} p/ calibração, "
      f"{len(negativos_report)} p/ relatório final | {len(load_test)} load_test")

TAUS = [2.5, 3, 3.5, 4, 5, 6, 8, 10, 15]
PESOS = {'nenhum': {0:1.0,1:1.0,2:1.0,3:1.0}, 'moderado': {0:1.0,1:1.0,2:0.7,3:0.4}, 'agressivo': {0:1.0,1:1.0,2:0.4,3:0.15}}

def entidades_do_servico(services_str):
    try:
        servs = _json.loads(services_str) if isinstance(services_str, str) else []
    except Exception:
        servs = []
    return [s.lower() for s in servs if s and s != '*']

ALL_ENTITIES = sbf['entity'].astype(str).unique()
ALL_ENTITIES_LOWER = np.array([e.lower() for e in ALL_ENTITIES])

def entidades_relevantes(services_str):
    servs = entidades_do_servico(services_str)
    if not servs:
        return None  # None = sem serviço específico (ex.: load_test, services=["*"]) -> universo inteiro
    mask = np.zeros(len(ALL_ENTITIES), dtype=bool)
    for s in servs:
        mask |= np.array([s in e for e in ALL_ENTITIES_LOWER])
    subset = set(ALL_ENTITIES[mask])
    return subset if subset else None

watched_entities = set()
for _, r in labels.iterrows():
    sub = entidades_relevantes(r['services'])
    if sub:
        watched_entities |= sub
print(f"{len(watched_entities)} entidades 'observadas' (união das entidades ligadas ao campo "
      f"`services` dos 20 rótulos), usadas como escopo do FPR, em vez do universo de "
      f"{sbf['entity'].nunique()} entidades monitoradas")

def dispara(tau, pesos, start, end, entity_subset=None):
    sub = sbf[(sbf['bts']>=start) & (sbf['bts']<=end)]
    if entity_subset is not None:
        sub = sub[sub['entity'].isin(entity_subset)]
    if sub.empty: return False
    w = sub['tier'].map(pesos).fillna(1.0)
    return ((sub['z'].abs() * w) > tau).any()

resultados_loio = []
for i in range(len(confiaveis)):
    held_out = confiaveis.iloc[i]
    held_scope = entidades_relevantes(held_out['services'])
    train = confiaveis.drop(index=i)
    best_cfg, best_score = None, (-2.0, 1e9)
    for tau in TAUS:
        for wname, wdict in PESOS.items():
            hits = sum(
                dispara(tau, wdict, r['ts_start'], r['ts_end_f'], entidades_relevantes(r['services']))
                for _, r in train.iterrows()
            )
            recall_tr = hits / len(train)
            fp = sum(dispara(tau, wdict, s, e, watched_entities) for s, e in negativos_calib)
            fpr_tr = fp / len(negativos_calib)
            meets = fpr_tr <= 0.10
            score = (recall_tr if meets else recall_tr - 1, -fpr_tr)
            if score > best_score:
                best_score, best_cfg = score, (tau, wname, wdict)
    tau, wname, wdict = best_cfg
    hit = dispara(tau, wdict, held_out['ts_start'], held_out['ts_end_f'], held_scope)
    resultados_loio.append({'held_out': held_out['id'], 'tau': tau, 'peso': wname, 'acertou': hit})

loio_df = pd.DataFrame(resultados_loio)
k, n = int(loio_df['acertou'].sum()), len(loio_df)
lo, hi = clopper_pearson(k, n)
print(loio_df.to_string(index=False))
print(f"\nRECALL (LOIO, 6 rótulos confiáveis): {k}/{n} = {k/n:.0%}  |  IC 95% Clopper-Pearson: [{lo:.0%}, {hi:.0%}]")
print(f"Estabilidade do threshold entre folds: {sorted(loio_df['tau'].unique())} "
      f"(se variar muito, é achado, não bug)")

# config final "de produção": moda entre os folds
tau_final = loio_df['tau'].mode()[0]
peso_final_nome = loio_df['peso'].mode()[0]
peso_final = PESOS[peso_final_nome]
fp_report = sum(dispara(tau_final, peso_final, s, e, watched_entities) for s, e in negativos_report)
lo_fpr, hi_fpr = wilson_ci(fp_report, len(negativos_report))
print(f"\nConfig final (moda dos folds): τ={tau_final}, peso={peso_final_nome}")
print(f"FPR (medido em negativos NÃO usados na calibração): {fp_report}/{len(negativos_report)} = "
      f"{fp_report/len(negativos_report):.1%}  |  IC 95% Wilson: [{lo_fpr:.1%}, {hi_fpr:.1%}]")

# Diagnóstico: quais séries especificamente causam a maior parte do falso-positivo?
culpados = {}
for s, e in negativos_report:
    sub = sbf[(sbf['bts']>=s) & (sbf['bts']<=e) & (sbf['entity'].isin(watched_entities))]
    w = sub['tier'].map(peso_final).fillna(1.0)
    disparou = sub[(sub['z'].abs()*w) > tau_final]
    for chave in disparou[['source','entity','metric']].drop_duplicates().itertuples(index=False):
        culpados[chave] = culpados.get(chave, 0) + 1

top_culpados = sorted(culpados.items(), key=lambda kv: -kv[1])[:8]
print(f"\nSéries que mais disparam falso-positivo (de {len(negativos_report)} janelas quietas):")
for (source, entity, metric), n in top_culpados:
    print(f"  {n:3d}/{len(negativos_report)} janelas | {source} | {entity} | {metric}")
print("\nSe poucas séries concentram a maioria dos disparos, o próximo passo é excluir ou "
      "transformar (ex.: log) essas métricas específicas, o que não é um problema do método em geral.")

In [ ]:
incidentes_reais = labels[labels['kind'] == 'incident'].copy()
incidentes_reais['familia'] = np.where(
    incidentes_reais['services'].str.contains('search-api', case=False, na=False),
    'search-api (dominante)', 'outra'
)

for fam, grp in incidentes_reais.groupby('familia'):
    hits = [dispara(tau_final, peso_final, r['ts_start'], r['ts_end_f'], entidades_relevantes(r['services']))
            for _, r in grp.iterrows()]
    if fam == 'search-api (dominante)':
        k, n = sum(hits), len(hits)
        lo, hi = clopper_pearson(k, n)
        print(f"{fam}: {k}/{n} = {k/n:.0%}  (IC 95%: [{lo:.0%},{hi:.0%}])")
    else:
        print(f"{fam}: {sum(hits)}/{len(hits)}, detalhe por incidente (n pequeno demais p/ taxa agregada):")
        for (_, r), h in zip(grp.iterrows(), hits):
            print(f"    {'OK ' if h else 'não'}  {r['id']}")

In [ ]:
flagged = sbf[(sbf['z'].abs() * sbf['tier'].map(peso_final).fillna(1.0)) > tau_final].sort_values(['source','entity','metric','bts']).copy()

episodios = []
for (source, entity, metric), g in flagged.groupby(['source','entity','metric'], observed=True):
    g = g.sort_values('bts')
    gap = g['bts'].diff().fillna(0) > 30*60*1000
    ep_id = gap.cumsum()
    for _, ep in g.groupby(ep_id):
        episodios.append({
            'source': source, 'entity': entity, 'metric': metric,
            'inicio': ep['bts'].min(), 'fim': ep['bts'].max(),
            'z_pico': ep['z'].abs().max(), 'duracao_buckets': len(ep),
            'tier': ep['tier'].iloc[0],
        })
ep_df = pd.DataFrame(episodios)
ep_df['score'] = ep_df['z_pico'] * ep_df['duracao_buckets'].clip(upper=8) * (1 + 0.3*(ep_df['tier']<=1))
ep_df = ep_df.sort_values('score', ascending=False).reset_index(drop=True)

search_api_entities = set(incidentes_reais.loc[incidentes_reais['familia']=='search-api (dominante)', 'entity']) \
    if 'entity' in incidentes_reais.columns else set()
search_api_known = incidentes_reais[incidentes_reais['familia']=='search-api (dominante)']

def rotula(row):
    janela = pd.Interval(row['inicio'], row['fim'], closed='both')
    for _, inc in incidentes_reais.iterrows():
        if pd.Interval(inc['ts_start'], inc['ts_end_f'], closed='both').overlaps(janela):
            return f"já rotulado: {inc['id']}"
    if 'search-api' in str(row['entity']).lower():
        return 'sugestão: padrão Search API (recorrente, alta confiança)'
    return 'candidato novo — revisão manual necessária'

ep_df['rotulo'] = ep_df.apply(rotula, axis=1)
ep_df['inicio_dt'] = to_dt(ep_df['inicio']); ep_df['fim_dt'] = to_dt(ep_df['fim'])

resumo_rotulo = ep_df['rotulo'].str.replace(r'já rotulado: .*', 'já rotulado (existente)', regex=True).value_counts()
print(f"{len(ep_df)} episódios encontrados no histórico inteiro:")
print(resumo_rotulo)
print(f"\nTop 10 candidatos novos (revisão manual), por score:")
novos = ep_df[ep_df['rotulo']=='candidato novo — revisão manual necessária']
novos[['inicio_dt','source','entity','metric','z_pico','duracao_buckets','score']].head(10)


In [ ]:
sugeridos_search_api = ep_df[ep_df['rotulo'].str.startswith('sugestão')]
n_positivos_search_api = len(search_api_known) + len(sugeridos_search_api)
PISO_FASE5 = 16
print(f"Rótulos totais utilizáveis pro padrão Search API após esta rodada: "
      f"{len(search_api_known)} confirmados + {len(sugeridos_search_api)} sugeridos pelo detector "
      f"= {n_positivos_search_api} (piso pra iniciar Fase 5: {PISO_FASE5}, sendo ao menos metade positiva/metade negativa confirmada)")
if n_positivos_search_api < PISO_FASE5:
    print("=> Fase 5 (classificador supervisionado) NÃO inicia nesta rodada — faltam confirmações "
          "humanas das sugestões acima. Isso é o comportamento correto, não uma falha.")
else:
    print("=> piso atingido; ainda assim, as sugestões do detector precisam de confirmação humana "
          "antes de virarem rótulo de treino (não tratar sugestão do próprio detector como verdade).")

In [ ]:
qoe_all = pd.read_csv(f'{DATA}/live_qoe_hourly.csv')
qoe_all['dt'] = to_dt(qoe_all['hour_ts'])
print(f"cobertura: {qoe_all['dt'].min()} até {qoe_all['dt'].max()} | {qoe_all['sinal'].nunique()} sinais")

trafego_mediano = qoe_all.groupby('sinal')['sessions'].median()
sinais_validos = trafego_mediano[trafego_mediano > 10].index
qoe = qoe_all[qoe_all['sinal'].isin(sinais_validos)].sort_values(['sinal', 'dt']).reset_index(drop=True)
print(f"{qoe['sinal'].nunique()} sinais com tráfego relevante (mediana de sessões acima de dez), "
      f"de {qoe_all['sinal'].nunique()} sinais totais, mantidos para o treinamento")

grp = qoe.groupby('sinal')['bufferratio_avg']
qoe['lag_1h'] = grp.shift(1)
qoe['lag_2h'] = grp.shift(2)
qoe['lag_3h'] = grp.shift(3)
qoe['roll_mean_3h'] = grp.transform(lambda s: s.shift(1).rolling(3).mean())
qoe['alvo'] = grp.shift(-1)
qoe['hora'] = qoe['dt'].dt.hour
qoe['hora_sin'] = np.sin(2*np.pi*qoe['hora']/24)
qoe['hora_cos'] = np.cos(2*np.pi*qoe['hora']/24)
qoe['dia_semana'] = qoe['dt'].dt.dayofweek

FEATURES = ['bufferratio_avg', 'lag_1h', 'lag_2h', 'lag_3h', 'roll_mean_3h',
            'sessions', 'errors', 'startup_errors', 'hora_sin', 'hora_cos', 'dia_semana']
dataset = qoe.dropna(subset=FEATURES + ['alvo']).copy()
print(f"{len(dataset)} observações utilizáveis, depois de remover as primeiras horas de cada sinal "
      f"(sem histórico suficiente para lag e média móvel) e a última hora de cada sinal (sem alvo futuro)")

In [ ]:
CORTE_QOE = pd.Timestamp('2026-08-01')
treino = dataset[dataset['dt'] < CORTE_QOE]
teste = dataset[dataset['dt'] >= CORTE_QOE]
print(f"treino: {len(treino)} observações (até {CORTE_QOE.date()}) | "
      f"teste: {len(teste)} observações (a partir de {CORTE_QOE.date()}), sem mistura temporal")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

modelo_qoe = RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                    random_state=42, n_jobs=-1)
modelo_qoe.fit(treino[FEATURES], treino['alvo'])
previsto = modelo_qoe.predict(teste[FEATURES])

rmse = mean_squared_error(teste['alvo'], previsto) ** 0.5
mae = mean_absolute_error(teste['alvo'], previsto)
r2 = r2_score(teste['alvo'], previsto)

media_treino = np.full(len(teste), treino['alvo'].mean())
rmse_media = mean_squared_error(teste['alvo'], media_treino) ** 0.5
persistencia = teste['bufferratio_avg'].to_numpy()
rmse_persistencia = mean_squared_error(teste['alvo'], persistencia) ** 0.5

print("Métricas de regressão no período de teste (dado nunca visto no treino):")
print(f"  RMSE do modelo: {rmse:.4f}")
print(f"  MAE do modelo: {mae:.4f}")
print(f"  R2 do modelo: {r2:.4f}")
print()
print("Comparação com dois baselines ingênuos, para dar sentido aos números acima:")
print(f"  RMSE prevendo sempre a média do treino: {rmse_media:.4f}")
print(f"  RMSE prevendo sempre o valor atual (persistência, o baseline forte em séries temporais): "
      f"{rmse_persistencia:.4f}")


In [ ]:
importancias = pd.Series(modelo_qoe.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
importancias.plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_title('Importância de cada variável na previsão do buffer ratio da próxima hora')
plt.tight_layout(); plt.show()

sinal_exemplo = teste.loc[teste['sessions'].idxmax(), 'sinal']
serie_exemplo = teste[teste['sinal'] == sinal_exemplo].sort_values('dt')
pred_exemplo = modelo_qoe.predict(serie_exemplo[FEATURES])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(serie_exemplo['dt'], serie_exemplo['alvo'], color='#4C72B0', label='valor real', marker='o', markersize=3)
ax.plot(serie_exemplo['dt'], pred_exemplo, color='#C44E52', label='previsto pelo modelo', marker='x', markersize=3)
ax.set_title(f'Buffer ratio real vs. previsto, sinal {sinal_exemplo} (período de teste)')
ax.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()